# Route Resilience — Phase III: Network Criticality (DeepLab Graphs)

**Team:** CanopyBreakers | **Leader:** Sathyagith V

Reads healed road graphs from Phase II (`graphs_deeplab/`), ranks **gatekeeper** nodes by betweenness centrality, runs **ablation** stress tests, and exports dashboard JSON + heatmap PNGs.

**Input:** `outputs/graphs_deeplab/{id}_graph.json`  
**Output:** `outputs/analysis_deeplab/{id}_criticality.json` + `outputs/analysis_viz_deeplab/{id}_criticality.png`

**Runtime:** Google Colab (CPU is fine — NetworkX graph analytics)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%matplotlib inline

import subprocess
subprocess.run(["pip", "install", "-q", "networkx", "numpy", "matplotlib"], check=True)
print("Libraries ready: networkx, numpy, matplotlib")

In [ ]:
import glob
import os

DRIVE_BASE = "/content/drive/MyDrive/RouteResilience"

GRAPH_DIR = os.path.join(DRIVE_BASE, "outputs/graphs_deeplab")
ANALYSIS_DIR = os.path.join(DRIVE_BASE, "outputs/analysis_deeplab")
VIS_DIR = os.path.join(DRIVE_BASE, "outputs/analysis_viz_deeplab")

TOP_K = 5
WEIGHT_ATTR = "length"  # edge field in Phase II graph JSON

os.makedirs(ANALYSIS_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)

graph_files = sorted(glob.glob(os.path.join(GRAPH_DIR, "*_graph.json")))
if not graph_files:
    raise FileNotFoundError(
        f"No graph JSON files found in {GRAPH_DIR}. Run Phase II (phase2_deeplab_graph_healing.ipynb) first."
    )

print(f"GRAPH_DIR:    {GRAPH_DIR}")
print(f"ANALYSIS_DIR: {ANALYSIS_DIR}")
print(f"VIS_DIR:      {VIS_DIR}")
print(f"Found {len(graph_files)} graph file(s):")
for p in graph_files:
    print(f"  - {os.path.basename(p)}")

In [ ]:
import json
import networkx as nx


def image_id_from_path(path):
    return os.path.basename(path).replace("_graph.json", "")


def load_graph_from_json(path, weight_attr=WEIGHT_ATTR):
    """Load Phase II graph JSON; map edge length -> NetworkX weight."""
    with open(path, "r") as f:
        data = json.load(f)

    G = nx.Graph()
    for n in data.get("nodes", []):
        G.add_node(
            n["id"],
            x=n["x"],
            y=n["y"],
            degree=n.get("degree"),
        )

    for e in data.get("edges", []):
        w = float(e.get(weight_attr, e.get("weight", 1.0)))
        G.add_edge(
            e["source"],
            e["target"],
            weight=w,
            length=w,
            coords=e.get("coords", []),
            healed=bool(e.get("healed", False)),
            edge_id=e.get("id", f"e{e['source']}_{e['target']}"),
        )

    image_id = data.get("image_id") or image_id_from_path(path)
    return G, image_id


def largest_component(G):
    """Return the largest connected subgraph (analysis target)."""
    if G.number_of_nodes() == 0:
        return G.copy()
    nodes = max(nx.connected_components(G), key=len)
    return G.subgraph(nodes).copy()


print("Graph loaders defined.")

In [ ]:
def compute_betweenness(G):
    """Normalized betweenness centrality using edge geometric length as weight."""
    if G.number_of_nodes() < 2:
        return {}
    return nx.betweenness_centrality(G, weight="weight", normalized=True)


def rank_gatekeepers(G, bc, top_k=TOP_K):
    """Rank nodes by betweenness; return top-k gatekeeper records for the dashboard."""
    ranked = sorted(bc.items(), key=lambda item: item[1], reverse=True)
  gatekeepers = []
  for rank, (node_id, score) in enumerate(ranked[:top_k], start=1):
      attrs = G.nodes[node_id]
      gatekeepers.append({
          "id": node_id,
          "betweenness": round(float(score), 4),
          "rank": rank,
          "x": int(attrs.get("x", 0)),
          "y": int(attrs.get("y", 0)),
      })
  return gatekeepers, ranked


print("Centrality functions defined.")

In [ ]:
def avg_path_length(G):
    """Mean shortest-path length over all reachable node pairs (weighted)."""
    nodes = list(G.nodes())
    n = len(nodes)
    if n < 2:
        return 0.0

    total = 0.0
    count = 0
    for i, u in enumerate(nodes):
        for v in nodes[i + 1 :]:
            if nx.has_path(G, u, v):
                total += nx.shortest_path_length(G, u, v, weight="weight")
                count += 1
    if count == 0:
        return float("inf")
    return total / count


def resilience_index(baseline_len, perturbed_len):
    """R = L_baseline / L_perturbed (values > 1 mean network degraded)."""
    if perturbed_len is None or perturbed_len <= 0 or perturbed_len == float("inf"):
        return 0.0
    return round(baseline_len / perturbed_len, 4)


def simulate_node_removal(G, node_id):
    """Remove one node and return perturbed graph + summary metrics."""
    G2 = G.copy()
    if node_id in G2:
        G2.remove_node(node_id)
    return {
        "graph": G2,
        "avg_path_length": avg_path_length(G2),
        "connected_components": nx.number_connected_components(G2),
    }


def run_ablation_study(G, ranked_nodes, baseline_len, top_k=TOP_K):
    """Ablate top-k gatekeepers one at a time."""
    results = []
    for rank, (node_id, bc_score) in enumerate(ranked_nodes[:top_k], start=1):
        sim = simulate_node_removal(G, node_id)
        perturbed_len = sim["avg_path_length"]
        results.append({
            "removed_node": node_id,
            "avg_path_length": perturbed_len,
            "connected_components": sim["connected_components"],
            "resilience_index": resilience_index(baseline_len, perturbed_len),
            "rank": rank,
            "betweenness": round(float(bc_score), 4),
        })
    return results


print("Resilience / ablation functions defined.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np


def edge_criticality(u, v, bc):
    return (bc.get(u, 0.0) + bc.get(v, 0.0)) / 2.0


def build_dashboard_json(image_id, G, gatekeepers, bc, baseline_len, ablation_results):
    """Assemble Phase IV-compatible criticality.json payload."""
    edges_out = []
    for u, v, data in G.edges(data=True):
        crit = edge_criticality(u, v, bc)
        edges_out.append({
            "id": data.get("edge_id", f"e{u}_{v}"),
            "source": u,
            "target": v,
            "criticality": round(float(crit), 4),
            "healed": bool(data.get("healed", False)),
            "coords": data.get("coords", []),
        })

    top = ablation_results[0] if ablation_results else {}
    baseline_cc = nx.number_connected_components(G)

    return {
        "image_id": image_id,
        "gatekeepers": gatekeepers,
        "edges": edges_out,
        "metrics": {
            "baseline_avg_path_length": round(float(baseline_len), 4),
            "perturbed_avg_path_length": round(float(top.get("avg_path_length", baseline_len)), 4),
            "resilience_index": top.get("resilience_index", 1.0),
            "baseline_connected_components": baseline_cc,
            "connected_components": top.get("connected_components", baseline_cc),
        },
        "ablation_study": ablation_results,
    }


def save_analysis(payload, analysis_dir=ANALYSIS_DIR):
    out_path = os.path.join(analysis_dir, f"{payload['image_id']}_criticality.json")
    with open(out_path, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"Saved analysis -> {out_path}")
    return out_path


def plot_criticality(G, bc, image_id, vis_dir=VIS_DIR):
    """Draw edge criticality heatmap + node betweenness sizing."""
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_facecolor("#1a1a2e")
    fig.patch.set_facecolor("#1a1a2e")

    crit_values = []
    edge_segments = []
    for u, v, data in G.edges(data=True):
        coords = data.get("coords", [])
        if len(coords) < 2:
            x0, y0 = G.nodes[u].get("x", 0), G.nodes[u].get("y", 0)
            x1, y1 = G.nodes[v].get("x", 0), G.nodes[v].get("y", 0)
            xs, ys = [x0, x1], [y0, y1]
        else:
            ys = [pt[0] for pt in coords]
            xs = [pt[1] for pt in coords]
        crit = edge_criticality(u, v, bc)
        crit_values.append(crit)
        edge_segments.append((xs, ys, crit, bool(data.get("healed", False))))

    vmax = max(crit_values) if crit_values else 1.0
    norm = mcolors.Normalize(vmin=0.0, vmax=max(vmax, 1e-6))
    cmap = cm.get_cmap("RdYlGn_r")

    for xs, ys, crit, healed in edge_segments:
        color = cmap(norm(crit))
        ls = "--" if healed else "-"
        ax.plot(xs, ys, color=color, linewidth=1.5 if not healed else 1.0, alpha=0.85, linestyle=ls)

    node_x, node_y, sizes, colors = [], [], [], []
    for node_id, attrs in G.nodes(data=True):
        node_x.append(attrs.get("x", 0))
        node_y.append(attrs.get("y", 0))
        bc_val = bc.get(node_id, 0.0)
        sizes.append(20 + 300 * bc_val)
        colors.append(bc_val)

    sc = ax.scatter(node_x, node_y, s=sizes, c=colors, cmap="plasma", edgecolors="white", linewidths=0.3, zorder=5)
    cbar = plt.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label("Edge criticality", color="white")
    cbar.ax.yaxis.set_tick_params(color="white")
    plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="white")

    ax.set_title(f"Criticality heatmap — {image_id}", color="white", fontsize=14)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")

    out_path = os.path.join(vis_dir, f"{image_id}_criticality.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, facecolor=fig.get_facecolor())
    plt.show()
    plt.close(fig)
    print(f"Saved heatmap -> {out_path}")
    return out_path


print("Export + visualization functions defined.")

In [ ]:
def analyze_graph(graph_path):
    G_full, image_id = load_graph_from_json(graph_path)
    G = largest_component(G_full)

    if G.number_of_nodes() < 2:
        print(f"[SKIP] {image_id}: largest component has < 2 nodes")
        return None

    bc = compute_betweenness(G)
    gatekeepers, ranked = rank_gatekeepers(G, bc, top_k=TOP_K)
    baseline_len = avg_path_length(G)
    ablation = run_ablation_study(G, ranked, baseline_len, top_k=TOP_K)

    payload = build_dashboard_json(image_id, G, gatekeepers, bc, baseline_len, ablation)
    save_analysis(payload)
    plot_criticality(G, bc, image_id)

    top = gatekeepers[0] if gatekeepers else {}
    worst = ablation[0] if ablation else {}
    return {
        "image_id": image_id,
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "baseline_L": round(baseline_len, 4),
        "top_gatekeeper": top.get("id", "—"),
        "top_bc": top.get("betweenness", 0.0),
        "resilience_R": worst.get("resilience_index", 0.0),
        "components_after_top_removal": worst.get("connected_components", 0),
    }


summary_rows = []
for graph_path in graph_files:
    print("\n" + "=" * 60)
    print(f"Analyzing {os.path.basename(graph_path)}")
    row = analyze_graph(graph_path)
    if row:
        summary_rows.append(row)

print("\n" + "=" * 60)
print("PHASE III SUMMARY")
print("=" * 60)

if summary_rows:
    headers = ["image_id", "nodes", "edges", "baseline_L", "top_gatekeeper", "top_bc", "resilience_R", "components_after_top_removal"]
    widths = {h: max(len(h), *(len(str(r[h])) for r in summary_rows)) for h in headers}
    header_line = " | ".join(h.ljust(widths[h]) for h in headers)
    print(header_line)
    print("-" * len(header_line))
    for row in summary_rows:
        print(" | ".join(str(row[h]).ljust(widths[h]) for h in headers))
else:
    print("No graphs were successfully analyzed.")

print(f"\nDone. {len(summary_rows)} / {len(graph_files)} graphs exported to {ANALYSIS_DIR}")